In [1]:
# Fusion (Spatial + FFT) ---
import os, cv2 , numpy as np, random
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
SEED=15
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

In [4]:
REAL_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/train"
FAKE_DIR = "/content/drive/My Drive/Research Project/StyleGAN2_7k/train"
IMG_SIZE=(128,128)

In [5]:
def load_spatial_and_fft(real_dir, fake_dir, img_size=(128,128), limit=None):
    Xs, Xf, y = [], [], []
    def proc_sp(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size).astype(np.float32)/255.0
        return img[..., None]
    def proc_fft(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size).astype(np.float32)/255.0
        F = np.fft.fft2(img); Fshift = np.fft.fftshift(F)
        mag = np.abs(Fshift); mag_log = np.log(mag+1e-8)
        return mag_log[..., None]
    reals = sorted(os.listdir(real_dir))
    fakes = sorted(os.listdir(fake_dir))
    if limit: reals, fakes = reals[:limit], fakes[:limit]
    for fn in reals:
        p = os.path.join(real_dir, fn)
        Xs.append(proc_sp(p)); Xf.append(proc_fft(p)); y.append(0)
    for fn in fakes:
        p = os.path.join(fake_dir, fn)
        Xs.append(proc_sp(p)); Xf.append(proc_fft(p)); y.append(1)
    return np.array(Xs), np.array(Xf), np.array(y)

In [6]:
X_sp, X_fft, y = load_spatial_and_fft(REAL_DIR, FAKE_DIR)
print("Spatial:", X_sp.shape, "FFT:", X_fft.shape, "labels:", y.shape)

Spatial: (10575, 128, 128, 1) FFT: (10575, 128, 128, 1) labels: (10575,)


In [12]:
def build_branch(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(16,(4,4),activation='relu')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64,activation='relu')(x)
    return models.Model(inp, x)

In [8]:
def build_fusion(input_shape):
    a = build_branch(input_shape)
    b = build_branch(input_shape)
    comb = layers.Concatenate()([a.output, b.output])
    x = layers.Dense(64, activation='relu')(comb)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model([a.input, b.input], out)

In [9]:
model = build_fusion((128,128,1))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
Xs_tr, Xs_te, Xf_tr, Xf_te, y_tr, y_te = train_test_split(X_sp, X_fft, y, test_size=0.2, random_state=SEED, stratify=y)

early_stop = EarlyStopping(monitor='val_loss', patience=9, restore_best_weights=True)
history = model.fit([Xs_tr, Xf_tr], y_tr, validation_data=([Xs_te, Xf_te], y_te), epochs=20, batch_size=35, callbacks=[early_stop], verbose=1)

print("Test eval:", model.evaluate([Xs_te, Xf_te], y_te, verbose=0))

Epoch 1/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 18s 43ms/step - accuracy: 0.8544 - loss: 0.3086 - val_accuracy: 0.9896 - val_loss: 0.0324
Epoch 2/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.9819 - loss: 0.0573 - val_accuracy: 0.9905 - val_loss: 0.0328
Epoch 3/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.9852 - loss: 0.0489 - val_accuracy: 0.9962 - val_loss: 0.0121
Epoch 4/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.9847 - loss: 0.0446 - val_accuracy: 0.9924 - val_loss: 0.0220
Epoch 5/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.9868 - loss: 0.0333 - val_accuracy: 0.9991 - val_loss: 0.0057
Epoch 6/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.9914 - loss: 0.0277 - val_accuracy: 0.9991 - val_loss: 0.0050
Epoch 7/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.9938 - loss: 0.0199 - val_accuracy: 0.9943 - val_loss: 0.0133
Epoch 8/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.9955 - loss: 0.0162 - val_acc

In [11]:
from sklearn.metrics import accuracy_score, classification_report

from google.colab import drive
drive.mount('/content/drive')

REAL_TEST_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/test"
FAKE_TEST_DIR = "/content/drive/My Drive/Research Project/StyleGAN3_7k/test"

Xs_test, Xf_test, y_test = load_spatial_and_fft(REAL_TEST_DIR, FAKE_TEST_DIR, img_size=IMG_SIZE, limit=None)

print("StyleGAN3 Test Shapes:", Xs_test.shape, Xf_test.shape, y_test.shape)


test_loss, test_acc = model.evaluate([Xs_test, Xf_test], y_test, verbose=1)
print("StyleGAN3 Test Loss:", test_loss)
print("StyleGAN3 Test Accuracy:", test_acc)


y_pred_probs = model.predict([Xs_test, Xf_test])
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

print("\nClassification Report (StyleGAN3 Test Set):")
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
StyleGAN3 Test Shapes: (3525, 128, 128, 1) (3525, 128, 128, 1) (3525,)
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9926 - loss: 0.0291
StyleGAN3 Test Loss: 0.045575618743896484
StyleGAN3 Test Accuracy: 0.9869503378868103
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step

Classification Report (StyleGAN3 Test Set):
              precision    recall  f1-score   support

        Real       0.98      1.00      0.99      1750
        Fake       1.00      0.98      0.99      1775

    accuracy                           0.99      3525
   macro avg       0.99      0.99      0.99      3525
weighted avg       0.99      0.99      0.99      3525

